# Train Naive Bayes Incident Classifier

This notebook trains a Naive Bayes department classifier using `data/processed/incidents_train.csv`.
The text is vectorized with `TfidfVectorizer`, trained with a Multinomial Naive Bayes classifier, and evaluated with a classification report and confusion matrix.


In [6]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "processed" / "incidents_train.csv").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed/incidents_train.csv"
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_DATASET_PATH = PROCESSED_DATA_DIR / "incidents_train.csv"
VALIDATION_DATASET_PATH = PROCESSED_DATA_DIR / "incidents_validation.csv"
TEST_DATASET_PATH = PROCESSED_DATA_DIR / "incidents_test.csv"
LABEL_MAPPING_PATH = PROCESSED_DATA_DIR / "department_label_mapping.csv"

TEXT_COLUMN = "feature_concatanation"
LABEL_COLUMN = "department_label"
LABEL_NAME_COLUMN = "assigned_department"
RANDOM_SEED = 42


In [7]:
train_df = pd.read_csv(TRAIN_DATASET_PATH)
validation_df = pd.read_csv(VALIDATION_DATASET_PATH)
test_df = pd.read_csv(TEST_DATASET_PATH)
label_mapping_df = pd.read_csv(LABEL_MAPPING_PATH).sort_values(LABEL_COLUMN)

required_columns = {TEXT_COLUMN, LABEL_COLUMN, LABEL_NAME_COLUMN}
for dataset_name, dataset_df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    missing_columns = required_columns - set(dataset_df.columns)
    if missing_columns:
        raise ValueError(
            f"{dataset_name} dataset is missing required columns: {sorted(missing_columns)}"
        )


def prepare_dataset(dataset_df):
    prepared_df = dataset_df[[TEXT_COLUMN, LABEL_COLUMN, LABEL_NAME_COLUMN]].dropna().copy()
    prepared_df[TEXT_COLUMN] = prepared_df[TEXT_COLUMN].astype(str).str.strip()
    prepared_df = prepared_df[prepared_df[TEXT_COLUMN] != ""]
    prepared_df[LABEL_COLUMN] = prepared_df[LABEL_COLUMN].astype(int)
    return prepared_df


train_df = prepare_dataset(train_df)
validation_df = prepare_dataset(validation_df)
test_df = prepare_dataset(test_df)

label_mapping = dict(
    zip(label_mapping_df[LABEL_COLUMN].astype(int), label_mapping_df[LABEL_NAME_COLUMN])
)

print(f"Loaded {len(train_df)} train rows from {TRAIN_DATASET_PATH}")
print(f"Loaded {len(validation_df)} validation rows from {VALIDATION_DATASET_PATH}")
print(f"Loaded {len(test_df)} test rows from {TEST_DATASET_PATH}")
print(f"Number of departments: {len(label_mapping)}")
train_df.head()


Loaded 2520 train rows from /home/lakshan/ssp/IMS/data/processed/incidents_train.csv
Loaded 540 validation rows from /home/lakshan/ssp/IMS/data/processed/incidents_validation.csv
Loaded 540 test rows from /home/lakshan/ssp/IMS/data/processed/incidents_test.csv
Number of departments: 12


,feature_concatanation,department_label,assigned_department
0,Ward Sister - Medical Please send porter team ...,11,Transport
1,Pharmacy Manager Antibiotic bulk store damp; h...,7,Quality Assurance department
2,"Labour Room Midwife Between cases, blood splas...",4,Housekeeping Department
3,Pharmacy Intern - Inpatient Glucometer strips ...,10,Supply Department
4,"Head, Physiotherapy TENS units due for annual ...",0,Biomedical Engineering


In [8]:
X_train = train_df[TEXT_COLUMN]
y_train = train_df[LABEL_COLUMN]
X_validation = validation_df[TEXT_COLUMN]
y_validation = validation_df[LABEL_COLUMN]
X_test = test_df[TEXT_COLUMN]
y_test = test_df[LABEL_COLUMN]

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_validation), len(X_test)],
        "source": [
            str(TRAIN_DATASET_PATH),
            str(VALIDATION_DATASET_PATH),
            str(TEST_DATASET_PATH),
        ],
    }
)


,split,rows,source
0,train,2520,/home/lakshan/ssp/IMS/data/processed/incidents...
1,validation,540,/home/lakshan/ssp/IMS/data/processed/incidents...
2,test,540,/home/lakshan/ssp/IMS/data/processed/incidents...


In [9]:
naive_bayes_classifier = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_features=20000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            MultinomialNB(alpha=0.5),
        ),
    ]
)

naive_bayes_classifier.fit(X_train, y_train)
print("Finished training TF-IDF + Multinomial Naive Bayes classifier")


Finished training TF-IDF + Multinomial Naive Bayes classifier


In [10]:
validation_predictions = naive_bayes_classifier.predict(X_validation)
test_predictions = naive_bayes_classifier.predict(X_test)

validation_accuracy = accuracy_score(y_validation, validation_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

ordered_labels = sorted(label_mapping)
target_names = [label_mapping[label] for label in ordered_labels]

print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nTest classification report:\n")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=ordered_labels,
        target_names=target_names,
        zero_division=0,
    )
)

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=ordered_labels),
    index=[f"actual_{label_mapping[label]}" for label in ordered_labels],
    columns=[f"pred_{label_mapping[label]}" for label in ordered_labels],
)

print("\nConfusion matrix:\n")
confusion_matrix_df


Validation accuracy: 0.9185
Test accuracy: 0.9352

Test classification report:

                              precision    recall  f1-score   support

      Biomedical Engineering       0.93      0.95      0.94        42
   Dietary and Food Services       0.98      0.89      0.93        45
         Facility Management       0.95      0.83      0.89        48
         Finance and Account       0.88      0.98      0.93        51
     Housekeeping Department       0.92      0.96      0.94        46
             Human Resources       0.93      0.93      0.93        45
                          IT       1.00      0.93      0.97        45
Quality Assurance department       0.94      0.97      0.95        32
                   Reception       0.94      0.98      0.96        45
                    Security       0.95      0.97      0.96        40
           Supply Department       0.92      0.89      0.91        55
                   Transport       0.92      0.96      0.94        46

        

,pred_Biomedical Engineering,pred_Dietary and Food Services,pred_Facility Management,pred_Finance and Account,pred_Housekeeping Department,pred_Human Resources,pred_IT,pred_Quality Assurance department,pred_Reception,pred_Security,pred_Supply Department,pred_Transport
actual_Biomedical Engineering,40,0,1,0,0,0,0,1,0,0,0,0
actual_Dietary and Food Services,0,40,1,0,0,1,0,0,0,0,3,0
actual_Facility Management,2,1,40,0,3,0,0,0,0,1,0,1
actual_Finance and Account,0,0,0,50,0,1,0,0,0,0,0,0
actual_Housekeeping Department,0,0,0,1,44,0,0,0,0,0,1,0
actual_Human Resources,0,0,0,3,0,42,0,0,0,0,0,0
actual_IT,1,0,0,1,0,0,42,1,0,0,0,0
actual_Quality Assurance department,0,0,0,0,0,0,0,31,0,0,0,1
actual_Reception,0,0,0,0,1,0,0,0,44,0,0,0
actual_Security,0,0,0,0,0,0,0,0,1,39,0,0
